In [52]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableBranch, RunnableLambda
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

In [53]:
load_dotenv()
model = ChatOpenAI(model='gpt-4.1', temperature=0.2, max_completion_tokens= 500)

In [54]:
from pydantic import BaseModel, Field
from typing import Literal

In [55]:
class sentiment_analyizer(BaseModel):
    sentiment : Literal['positive','negative','neutral'] = Field(description='Sentiment of the text')
    feedback : str = Field(description='Generate the feedback according to sentiment')

In [56]:
parser = PydanticOutputParser(pydantic_object=sentiment_analyizer)

In [57]:
prompt1 = PromptTemplate(
    template='Find the sentiment of the below text \n {text} \{format_instruction}',
    input_variables=['text'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

<>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
/var/folders/94/2_5s7zqj0cqd50nz6b9khlxh0000gp/T/ipykernel_8853/880341557.py:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  template='Find the sentiment of the below text \n {text} \{format_instruction}',


In [58]:
comment = '''Oh wow, that’s just brilliant — if the goal was to do the absolute bare minimum and still act proud about it. Truly impressive how consistently disappointing this turned out to be. But hey, at least expectations can’t get any lower from here.'''

In [59]:
sentiment_chain = RunnableSequence(prompt1, model, parser)

In [60]:
prompt2 = PromptTemplate(
    template='generate a revert for the below negative feedback \n {comment_reply}',
    input_variables=['comment_reply']
)

In [61]:
prompt3 = PromptTemplate(
    template='generate a revert for the below positive feedback \n {comment_reply}',
    input_variables=['comment_reply']
)

In [62]:
prompt4 = PromptTemplate(
    template='generate a revert for the below neutral feedback \n {comment_reply}',
    input_variables=['comment_reply']
)

In [63]:
parser2 = StrOutputParser()

In [64]:
conditional_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', prompt3 | model | parser2),
    (lambda x: x.sentiment == 'negative',  prompt2 | model | parser2),
    (lambda x: x.sentiment == 'neutral', prompt4 | model | parser2), 
    RunnableLambda(lambda x: 'Not found')
)

In [65]:
final_chain = sentiment_chain | conditional_chain

In [66]:
final_chain.invoke({'text':comment})

'Certainly! Here’s a professional and empathetic response (revert) to the negative feedback you provided:\n\n---\n\nThank you for sharing your feedback. We’re sorry to hear that your experience did not meet your expectations and that you feel disappointed with the outcome. Your comments are important to us, and we take them seriously as we strive to improve our services. If you have any specific suggestions or further details to share, please let us know—we value your input and are committed to making things better moving forward.\n\n---'